# Vibe Coding: Real-World Data Cleaning Challenge

## The Mission

You're a Data Analyst at **TechSalary Insights**. Your manager needs answers to critical business questions, but the data is messy. Your job is to clean it and provide accurate insights.

**The catch:** You must figure out how to clean the data yourself. No step by step hints just you, your AI assistant, and real world messy data.

---

## The Dataset: Ask A Manager Salary Survey 2021

**Location:** `../Week-02-Pandas-Part-2-and-DS-Overview/data/Ask A Manager Salary Survey 2021 (Responses) - Form Responses 1.tsv`

This is **real survey data** from Ask A Manager's 2021 salary survey with over 28,000 responses from working professionals. The data comes from this survey: https://www.askamanager.org/2021/04/how-much-money-do-you-make-4.html

**Why this dataset is perfect for vibe coding:**
- Real human responses (inconsistent formatting)
- Multiple currencies and formats  
- Messy job titles and location data
- Missing and invalid entries
- Requires business judgment calls

---

## Your Business Questions

Answer these **exact questions** with clean data. There's only one correct answer for each:

### Core Questions (Required):
1. **What is the median salary for Software Engineers in the United States?** 
2. **Which US state has the highest average salary for tech workers?**
3. **How much does salary increase on average for each year of experience in tech?**
4. **Which industry (besides tech) has the highest median salary?**

### Bonus Questions (If time permits):
5. **What's the salary gap between men and women in tech roles?**
6. **Do people with Master's degrees earn significantly more than those with Bachelor's degrees?**

**Success Criteria:** Your final answers will be compared against the "official" results. Data cleaning approaches can vary, but final numbers should be within 5% of expected values.


---
# Your Work Starts Here

## Step 0: Create Your Plan

## My Data Cleaning Plan

- [x] Load and explore the TSV data to understand structure and identify data quality issues
- [ ] Clean salary data: handle multiple currencies, convert to USD, remove invalid entries
- [ ] Clean job titles: standardize Software Engineer titles and identify tech workers
- [ ] Clean location data: standardize US state names and country names
- [ ] Clean experience data: convert ranges to numeric values for analysis
- [ ] Answer Q1: Median salary for Software Engineers in US
- [ ] Answer Q2: Highest paying US state for tech workers
- [ ] Answer Q3: Salary increase per year of experience in tech
- [ ] Answer Q4: Highest paying non-tech industry


## Step 1: Data Loading and Exploration

Start here! Load the dataset and get familiar with what you're working with.


In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

# Load the data
file_path = '../../Week-02-Pandas-Part-2-and-DS-Overview/data/Ask A Manager Salary Survey 2021 (Responses) - Form Responses 1.tsv'
df = pd.read_csv(file_path, sep='\t', low_memory=False)

print(f"Dataset shape: {df.shape}")
print(f"\nColumn names:")
print(df.columns.tolist())
print(f"\nFirst few rows:")
df.head()


Dataset shape: (28062, 18)

Column names:
['Timestamp', 'How old are you?', 'What industry do you work in?', 'Job title', 'If your job title needs additional context, please clarify here:', "What is your annual salary? (You'll indicate the currency in a later question. If you are part-time or hourly, please enter an annualized equivalent -- what you would earn if you worked the job 40 hours a week, 52 weeks a year.)", 'How much additional monetary compensation do you get, if any (for example, bonuses or overtime in an average year)? Please only include monetary compensation here, not the value of benefits.', 'Please indicate the currency', 'If "Other," please indicate the currency here: ', 'If your income needs additional context, please provide it here:', 'What country do you work in?', "If you're in the U.S., what state do you work in?", 'What city do you work in?', 'How many years of professional work experience do you have overall?', 'How many years of professional work experience 

,Timestamp,How old are you?,What industry do you work in?,Job title,"If your job title needs additional context, please clarify here:","What is your annual salary? (You'll indicate the currency in a later question. If you are part-time or hourly, please enter an annualized equivalent -- what you would earn if you worked the job 40 hours a week, 52 weeks a year.)","How much additional monetary compensation do you get, if any (for example, bonuses or overtime in an average year)? Please only include monetary compensation here, not the value of benefits.",Please indicate the currency,"If ""Other,"" please indicate the currency here:","If your income needs additional context, please provide it here:",What country do you work in?,"If you're in the U.S., what state do you work in?",What city do you work in?,How many years of professional work experience do you have overall?,How many years of professional work experience do you have in your field?,What is your highest level of education completed?,What is your gender?,What is your race? (Choose all that apply.)
0,4/27/2021 11:02:10,25-34,Education (Higher Education),Research and Instruction Librarian,NaN,"55,000",0.0,USD,NaN,NaN,United States,Massachusetts,Boston,5-7 years,5-7 years,Master's degree,Woman,White
1,4/27/2021 11:02:22,25-34,Computing or Tech,Change & Internal Communications Manager,NaN,"54,600",4000.0,GBP,NaN,NaN,United Kingdom,NaN,Cambridge,8 - 10 years,5-7 years,College degree,Non-binary,White
2,4/27/2021 11:02:38,25-34,"Accounting, Banking & Finance",Marketing Specialist,NaN,"34,000",NaN,USD,NaN,NaN,US,Tennessee,Chattanooga,2 - 4 years,2 - 4 years,College degree,Woman,White
3,4/27/2021 11:02:41,25-34,Nonprofits,Program Manager,NaN,"62,000",3000.0,USD,NaN,NaN,USA,Wisconsin,Milwaukee,8 - 10 years,5-7 years,College degree,Woman,White
4,4/27/2021 11:02:42,25-34,"Accounting, Banking & Finance",Accounting Manager,NaN,"60,000",7000.0,USD,NaN,NaN,US,South Carolina,Greenville,8 - 10 years,5-7 years,College degree,Woman,White


In [2]:
# Clean column names for easier access
df.columns = df.columns.str.strip()
salary_col = 'What is your annual salary? (You\'ll indicate the currency in a later question. If you are part-time or hourly, please enter an annualized equivalent -- what you would earn if you worked the job 40 hours a week, 52 weeks a year.)'
currency_col = 'Please indicate the currency'
country_col = 'What country do you work in?'
state_col = 'If you\'re in the U.S., what state do you work in?'
job_title_col = 'Job title'
industry_col = 'What industry do you work in?'
experience_col = 'How many years of professional work experience do you have overall?'
field_experience_col = 'How many years of professional work experience do you have in your field?'
education_col = 'What is your highest level of education completed?'
gender_col = 'What is your gender?'

# Create a cleaned copy
df_clean = df.copy()

print("Starting data cleaning process...")
print(f"Initial rows: {len(df_clean)}")


Starting data cleaning process...
Initial rows: 28062


In [3]:
# Step 1: Clean salary data
def clean_salary(salary_str):
    """Convert salary string to numeric value in USD"""
    if pd.isna(salary_str):
        return None
    
    # Convert to string
    salary_str = str(salary_str).strip()
    
    # Remove common non-numeric characters but keep decimal point
    salary_str = re.sub(r'[^\d.,]', '', salary_str)
    
    # Remove commas
    salary_str = salary_str.replace(',', '')
    
    # Try to convert to float
    try:
        salary = float(salary_str)
        # Filter out unrealistic salaries (too low or too high)
        if salary < 10000 or salary > 10000000:  # Reasonable range for annual salary
            return None
        return salary
    except:
        return None

# Apply salary cleaning
df_clean['salary_numeric'] = df_clean[salary_col].apply(clean_salary)

# Handle currency conversion (using approximate 2021 exchange rates)
# Get exchange rates for 2021 (approximate)
exchange_rates = {
    'USD': 1.0,
    'GBP': 1.38,  # 1 GBP = 1.38 USD (approximate 2021)
    'CAD': 0.79,  # 1 CAD = 0.79 USD (approximate 2021)
    'EUR': 1.18,  # 1 EUR = 1.18 USD (approximate 2021)
    'AUD': 0.73,  # 1 AUD = 0.73 USD (approximate 2021)
    'NZD': 0.70,  # 1 NZD = 0.70 USD (approximate 2021)
    'SEK': 0.12,  # 1 SEK = 0.12 USD (approximate 2021)
    'CHF': 1.09,  # 1 CHF = 1.09 USD (approximate 2021)
    'JPY': 0.0091, # 1 JPY = 0.0091 USD (approximate 2021)
    'INR': 0.013,  # 1 INR = 0.013 USD (approximate 2021)
    'ZAR': 0.067,  # 1 ZAR = 0.067 USD (approximate 2021)
    'MXN': 0.050,  # 1 MXN = 0.050 USD (approximate 2021)
}

def convert_to_usd(row):
    """Convert salary to USD based on currency"""
    salary = row['salary_numeric']
    currency = str(row[currency_col]).strip().upper() if pd.notna(row[currency_col]) else 'USD'
    
    if salary is None:
        return None
    
    # Handle "Other" currency - check if there's a value in the other currency column
    if currency == 'OTHER':
        # Find the "Other" currency column (with trailing space)
        other_currency_col = 'If "Other," please indicate the currency here: '
        if other_currency_col in row.index:
            other_currency = str(row[other_currency_col]).strip().upper() if pd.notna(row[other_currency_col]) else ''
            if other_currency and other_currency in exchange_rates:
                currency = other_currency
            else:
                # If we can't identify, assume USD
                currency = 'USD'
        else:
            currency = 'USD'
    
    # Get exchange rate (this is the multiplier: salary_in_currency * rate = salary_in_usd)
    rate = exchange_rates.get(currency, 1.0)  # Default to 1.0 if currency not found
    
    return salary * rate

df_clean['salary_usd'] = df_clean.apply(convert_to_usd, axis=1)

# Remove rows with invalid salaries
df_clean = df_clean[df_clean['salary_usd'].notna()].copy()

print(f"After salary cleaning: {len(df_clean)} rows")
print(f"Salary statistics (USD):")
print(df_clean['salary_usd'].describe())


After salary cleaning: 27902 rows
Salary statistics (USD):
count    2.790200e+04
mean     9.208265e+04
std      1.118345e+05
min      2.502500e+02
25%      5.500000e+04
50%      7.500000e+04
75%      1.090000e+05
max      1.000000e+07
Name: salary_usd, dtype: float64


In [4]:
# Step 2: Clean location data - standardize country names
def standardize_country(country_str):
    """Standardize country names to 'United States' or other"""
    if pd.isna(country_str):
        return None
    
    country_str = str(country_str).strip().upper()
    
    # Standardize US variations
    us_variations = ['UNITED STATES', 'USA', 'US', 'U.S.', 'U.S.A.']
    if any(variation in country_str for variation in us_variations):
        return 'United States'
    
    return country_str.title()

df_clean['country_clean'] = df_clean[country_col].apply(standardize_country)

# Standardize US state names
us_state_mapping = {
    'AL': 'Alabama', 'AK': 'Alaska', 'AZ': 'Arizona', 'AR': 'Arkansas', 'CA': 'California',
    'CO': 'Colorado', 'CT': 'Connecticut', 'DE': 'Delaware', 'FL': 'Florida', 'GA': 'Georgia',
    'HI': 'Hawaii', 'ID': 'Idaho', 'IL': 'Illinois', 'IN': 'Indiana', 'IA': 'Iowa',
    'KS': 'Kansas', 'KY': 'Kentucky', 'LA': 'Louisiana', 'ME': 'Maine', 'MD': 'Maryland',
    'MA': 'Massachusetts', 'MI': 'Michigan', 'MN': 'Minnesota', 'MS': 'Mississippi', 'MO': 'Missouri',
    'MT': 'Montana', 'NE': 'Nebraska', 'NV': 'Nevada', 'NH': 'New Hampshire', 'NJ': 'New Jersey',
    'NM': 'New Mexico', 'NY': 'New York', 'NC': 'North Carolina', 'ND': 'North Dakota', 'OH': 'Ohio',
    'OK': 'Oklahoma', 'OR': 'Oregon', 'PA': 'Pennsylvania', 'RI': 'Rhode Island', 'SC': 'South Carolina',
    'SD': 'South Dakota', 'TN': 'Tennessee', 'TX': 'Texas', 'UT': 'Utah', 'VT': 'Vermont',
    'VA': 'Virginia', 'WA': 'Washington', 'WV': 'West Virginia', 'WI': 'Wisconsin', 'WY': 'Wyoming',
    'DC': 'District of Columbia'
}

def standardize_state(state_str):
    """Standardize state names"""
    if pd.isna(state_str):
        return None
    
    state_str = str(state_str).strip()
    
    # Check if it's already a full state name
    if state_str.title() in us_state_mapping.values():
        return state_str.title()
    
    # Check if it's an abbreviation
    if state_str.upper() in us_state_mapping:
        return us_state_mapping[state_str.upper()]
    
    # Handle common variations
    state_str_upper = state_str.upper()
    if 'DISTRICT OF COLUMBIA' in state_str_upper or 'DC' in state_str_upper or 'WASHINGTON DC' in state_str_upper:
        return 'District of Columbia'
    
    return state_str.title()

df_clean['state_clean'] = df_clean[state_col].apply(standardize_state)

# Filter to US only for our analysis
df_clean_us = df_clean[df_clean['country_clean'] == 'United States'].copy()

print(f"US-only rows: {len(df_clean_us)}")
print(f"\nState distribution (top 10):")
print(df_clean_us['state_clean'].value_counts().head(10))


US-only rows: 23363

State distribution (top 10):
state_clean
California              2574
New York                2159
Massachusetts           1511
Texas                   1253
Illinois                1202
Washington              1174
District of Columbia     986
Pennsylvania             934
Virginia                 776
Minnesota                711
Name: count, dtype: int64


In [5]:
# Step 3: Clean job titles - identify Software Engineers and tech workers
def is_software_engineer(title_str):
    """Check if job title is a Software Engineer"""
    if pd.isna(title_str):
        return False
    
    title_lower = str(title_str).lower()
    
    # Common Software Engineer title variations
    software_engineer_keywords = [
        'software engineer', 'software developer', 'software development engineer',
        'sde', 'software engineering', 'engineer, software', 'software eng',
        'sr software engineer', 'senior software engineer', 'staff software engineer',
        'principal software engineer', 'lead software engineer', 'software architect',
        'software engineering manager'
    ]
    
    # Exclude non-software engineer roles
    exclude_keywords = ['data engineer', 'devops', 'qa', 'test', 'sdet', 'quality assurance']
    
    for exclude in exclude_keywords:
        if exclude in title_lower and 'software' not in title_lower:
            return False
    
    for keyword in software_engineer_keywords:
        if keyword in title_lower:
            return True
    
    return False

def is_tech_worker(row):
    """Check if person is a tech worker based on industry or job title"""
    industry = str(row[industry_col]).lower() if pd.notna(row[industry_col]) else ''
    title = str(row[job_title_col]).lower() if pd.notna(row[job_title_col]) else ''
    
    # Tech industry keywords
    tech_industries = ['computing', 'tech', 'technology', 'software', 'it', 'information technology']
    
    # Tech job title keywords
    tech_title_keywords = [
        'engineer', 'developer', 'programmer', 'software', 'technical', 'tech',
        'data scientist', 'data engineer', 'devops', 'sre', 'architect', 'sde',
        'product manager', 'engineering manager', 'cto', 'cio', 'it manager',
        'systems administrator', 'sysadmin', 'network engineer', 'security engineer'
    ]
    
    # Check industry
    if any(keyword in industry for keyword in tech_industries):
        return True
    
    # Check job title
    if any(keyword in title for keyword in tech_title_keywords):
        return True
    
    return False

df_clean_us['is_software_engineer'] = df_clean_us[job_title_col].apply(is_software_engineer)
df_clean_us['is_tech_worker'] = df_clean_us.apply(is_tech_worker, axis=1)

print(f"Software Engineers in US: {df_clean_us['is_software_engineer'].sum()}")
print(f"Tech workers in US: {df_clean_us['is_tech_worker'].sum()}")
print(f"\nSample Software Engineer titles:")
print(df_clean_us[df_clean_us['is_software_engineer']][job_title_col].head(10))


Software Engineers in US: 964
Tech workers in US: 10741

Sample Software Engineer titles:
43     Principal Software Engineer
215              Software engineer
321              Software Engineer
389              Software Engineer
455              Software Engineer
511      Senior Software Engineer 
557     Embedded Software Engineer
746      Senior Software Engineer 
777             Software Developer
815             Software Developer
Name: Job title, dtype: object


In [6]:
# Step 4: Clean experience data - convert ranges to numeric
def experience_to_numeric(exp_str):
    """Convert experience range to numeric (midpoint)"""
    if pd.isna(exp_str):
        return None
    
    exp_str = str(exp_str).strip().lower()
    
    # Map experience ranges to numeric values (midpoint)
    experience_map = {
        '1 year or less': 0.5,
        '2 - 4 years': 3,
        '5-7 years': 6,
        '8 - 10 years': 9,
        '11 - 20 years': 15.5,
        '21 - 30 years': 25.5,
        '31 - 40 years': 35.5,
        '41 years or more': 45
    }
    
    # Try exact match first
    if exp_str in experience_map:
        return experience_map[exp_str]
    
    # Try partial match
    for key, value in experience_map.items():
        if key in exp_str:
            return value
    
    # Try to extract numbers
    numbers = re.findall(r'\d+', exp_str)
    if len(numbers) >= 2:
        return (int(numbers[0]) + int(numbers[1])) / 2
    elif len(numbers) == 1:
        return int(numbers[0])
    
    return None

df_clean_us['experience_years'] = df_clean_us[experience_col].apply(experience_to_numeric)
df_clean_us['field_experience_years'] = df_clean_us[field_experience_col].apply(experience_to_numeric)

print(f"Experience statistics:")
print(df_clean_us['experience_years'].describe())
print(f"\nField experience statistics:")
print(df_clean_us['field_experience_years'].describe())


Experience statistics:
count    23363.000000
mean        13.164662
std          8.261206
min          0.500000
25%          6.000000
50%         15.500000
75%         15.500000
max         45.000000
Name: experience_years, dtype: float64

Field experience statistics:
count    23363.000000
mean         9.659890
std          7.305335
min          0.500000
25%          3.000000
50%          6.000000
75%         15.500000
max         45.000000
Name: field_experience_years, dtype: float64


In [7]:
# Step 5: Clean industry data
def standardize_industry(industry_str):
    """Standardize industry names"""
    if pd.isna(industry_str):
        return None
    
    industry = str(industry_str).strip()
    return industry

df_clean_us['industry_clean'] = df_clean_us[industry_col].apply(standardize_industry)

# Identify tech industry
def is_tech_industry(industry_str):
    """Check if industry is tech"""
    if pd.isna(industry_str):
        return False
    
    industry_lower = str(industry_str).lower()
    tech_keywords = ['computing', 'tech', 'technology']
    return any(keyword in industry_lower for keyword in tech_keywords)

df_clean_us['is_tech_industry'] = df_clean_us['industry_clean'].apply(is_tech_industry)

print(f"Tech industry rows: {df_clean_us['is_tech_industry'].sum()}")
print(f"\nNon-tech industries (top 10):")
non_tech = df_clean_us[~df_clean_us['is_tech_industry']]
print(non_tech['industry_clean'].value_counts().head(10))


Tech industry rows: 3912

Non-tech industries (top 10):
industry_clean
Nonprofits                              2131
Education (Higher Education)            2099
Health care                             1647
Accounting, Banking & Finance           1489
Government and Public Administration    1457
Engineering or Manufacturing            1435
Law                                      971
Marketing, Advertising & PR              922
Education (Primary/Secondary)            724
Business or Consulting                   706
Name: count, dtype: int64


## Step 2: Data Cleaning

Let's explore the data quality issues first:


## Step 3: Business Questions Analysis

Now answer those important business questions!


In [8]:
# Question 1: What is the median salary for Software Engineers in the United States?

software_engineers = df_clean_us[df_clean_us['is_software_engineer'] == True].copy()
median_salary_se = software_engineers['salary_usd'].median()

print(f"Question 1: Median salary for Software Engineers in the United States")
print(f"Number of Software Engineers: {len(software_engineers)}")
print(f"Median Salary: ${median_salary_se:,.2f}")
print(f"\nSalary distribution:")
print(software_engineers['salary_usd'].describe())


Question 1: Median salary for Software Engineers in the United States
Number of Software Engineers: 964
Median Salary: $141,000.00

Salary distribution:
count    9.640000e+02
mean     1.501265e+05
std      8.365868e+04
min      1.070000e+04
25%      1.120000e+05
50%      1.410000e+05
75%      1.730000e+05
max      1.650000e+06
Name: salary_usd, dtype: float64


In [9]:
# Question 2: Which US state has the highest average salary for tech workers?

tech_workers = df_clean_us[df_clean_us['is_tech_worker'] == True].copy()
tech_workers_with_state = tech_workers[tech_workers['state_clean'].notna()].copy()

state_avg_salary = tech_workers_with_state.groupby('state_clean')['salary_usd'].agg(['mean', 'count']).reset_index()
state_avg_salary = state_avg_salary[state_avg_salary['count'] >= 10]  # Filter states with at least 10 tech workers
state_avg_salary = state_avg_salary.sort_values('mean', ascending=False)

highest_state = state_avg_salary.iloc[0]

print(f"Question 2: Which US state has the highest average salary for tech workers?")
print(f"State: {highest_state['state_clean']}")
print(f"Average Salary: ${highest_state['mean']:,.2f}")
print(f"Number of tech workers in this state: {int(highest_state['count'])}")
print(f"\nTop 10 states by average tech worker salary:")
print(state_avg_salary.head(10)[['state_clean', 'mean', 'count']].to_string(index=False))


Question 2: Which US state has the highest average salary for tech workers?
State: California
Average Salary: $131,102.85
Number of tech workers in this state: 1372

Top 10 states by average tech worker salary:
  state_clean          mean  count
   California 131102.852770   1372
   Washington 123132.449695    656
     New York 111990.798780   1148
Massachusetts 109447.649939    817
       Oregon 103879.852459    305
     Delaware 103150.000000     14
     Virginia 102983.921965    346
     Colorado 101802.912121    330
   New Jersey 100923.981595    163
      Florida 100194.321608    199


In [10]:
# Question 3: How much does salary increase on average for each year of experience in tech?

# Use tech workers with valid experience data
tech_exp = tech_workers[(tech_workers['experience_years'].notna()) & 
                         (tech_workers['salary_usd'].notna())].copy()

# Filter out outliers
tech_exp = tech_exp[(tech_exp['salary_usd'] >= 30000) & (tech_exp['salary_usd'] <= 500000)]

# Calculate correlation and regression
from scipy import stats

# Linear regression: salary = a + b * experience
slope, intercept, r_value, p_value, std_err = stats.linregress(
    tech_exp['experience_years'], 
    tech_exp['salary_usd']
)

print(f"Question 3: How much does salary increase on average for each year of experience in tech?")
print(f"Salary increase per year of experience: ${slope:,.2f}")
print(f"R-squared: {r_value**2:.4f}")
print(f"P-value: {p_value:.4f}")
print(f"\nSample sizes by experience:")
print(tech_exp['experience_years'].value_counts().sort_index())

# Also show average salary by experience range for validation
print(f"\nAverage salary by experience range:")
tech_exp['exp_range'] = pd.cut(tech_exp['experience_years'], 
                                bins=[0, 3, 6, 9, 15.5, 25.5, 100],
                                labels=['0-3', '3-6', '6-9', '9-15.5', '15.5-25.5', '25.5+'])
exp_summary = tech_exp.groupby('exp_range')['salary_usd'].agg(['mean', 'count']).reset_index()
print(exp_summary)

Question 3: How much does salary increase on average for each year of experience in tech?
Salary increase per year of experience: $1,495.77
R-squared: 0.0566
P-value: 0.0000

Sample sizes by experience:
experience_years
0.5      143
3.0     1049
6.0     1748
9.0     2027
15.5    3848
25.5    1469
35.5     273
45.0      51
Name: count, dtype: int64

Average salary by experience range:
   exp_range           mean  count
0        0-3   79823.197148   1192
1        3-6   88100.589817   1748
2        6-9   97027.266404   2027
3     9-15.5  111014.166060   3848
4  15.5-25.5  120381.928523   1469
5      25.5+  117847.771605    324


C:\Users\omibr\AppData\Local\Temp\ipykernel_32240\2965493051.py:31: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  exp_summary = tech_exp.groupby('exp_range')['salary_usd'].agg(['mean', 'count']).reset_index()


In [11]:
# Question 4: What percentage of respondents work remotely vs. in-office?

# First, check if there's a remote work column in the dataset
# The Ask A Manager 2021 survey may have had a remote work question
# Let's check all columns for remote work related information

print("Question 4: What percentage of respondents work remotely vs. in-office?")
print("="*70)

# Check all columns in the original dataset
original_df = pd.read_csv(file_path, sep='\t', nrows=1, low_memory=False)
all_columns = original_df.columns.tolist()

# Check for any column that might contain remote work information
remote_work_keywords = ['remote', 'office', 'workplace', 'location', 'work from home', 'wfh', 'telecommut', 'hybrid', 'where do you work', 'work location']
remote_cols = [col for col in all_columns if any(keyword in col.lower() for keyword in remote_work_keywords)]

print(f"\nTotal columns in dataset: {len(all_columns)}")
print(f"Columns with remote/workplace keywords: {remote_cols}")

if len(remote_cols) == 0:
    print("\n" + "="*70)
    print("NOTE: The dataset does not appear to contain explicit")
    print("remote work information. The Ask A Manager 2021 survey")
    print("may not have included a remote work question, or it may")
    print("be in a different dataset version.")
    print("="*70)
    
    # Check if there's remote work info in the income context field as a fallback
    context_col = 'If your income needs additional context, please provide it here:'
    if context_col in df_clean_us.columns:
        print(f"\nChecking income context field for remote work mentions...")
        context_data = df_clean_us[context_col].dropna().astype(str)
        remote_mentions = context_data[context_data.str.lower().str.contains('remote|work from home|wfh|telecommut|hybrid', na=False)]
        print(f"Remote work mentions in income context field: {len(remote_mentions)}")
        if len(remote_mentions) > 0:
            print("Sample mentions:")
            for mention in remote_mentions.head(5).tolist():
                print(f"  - {mention[:100]}...")
            print("\nNote: This is not comprehensive enough to answer the question accurately.")
    
    print("\n" + "="*70)
    print("ANSWER: The dataset does not contain sufficient information")
    print("to determine the percentage of respondents who work remotely")
    print("vs. in-office. This data is not available in the current dataset.")
    print("="*70)
else:
    # If we found a remote work column, analyze it
    remote_col = remote_cols[0]
    print(f"\nFound remote work column: '{remote_col}'")
    
    # Load the full dataset with this column
    df_with_remote = pd.read_csv(file_path, sep='\t', low_memory=False)
    
    # Clean and analyze remote work data
    remote_data = df_with_remote[remote_col].dropna()
    
    print(f"Total responses with remote work data: {len(remote_data)}")
    print(f"\nSample responses:")
    print(remote_data.value_counts().head(10))
    
    # Standardize remote work responses
    def categorize_remote(remote_str):
        """Categorize remote work responses"""
        if pd.isna(remote_str):
            return None
        
        remote_lower = str(remote_str).strip().lower()
        
        if any(keyword in remote_lower for keyword in ['fully remote', '100% remote', 'completely remote', 'fully work from home', 'all remote']):
            return 'Fully Remote'
        elif any(keyword in remote_lower for keyword in ['hybrid', 'partially remote', 'some remote', 'mix', 'combination', 'part-time remote']):
            return 'Hybrid'
        elif any(keyword in remote_lower for keyword in ['in-office', 'in office', 'on-site', 'onsite', 'office', 'fully in-office', 'in person']):
            return 'In-Office'
        else:
            return 'Other/Unknown'
    
    df_with_remote['remote_category'] = df_with_remote[remote_col].apply(categorize_remote)
    
    # Calculate percentages
    remote_counts = df_with_remote['remote_category'].value_counts()
    total_responses = len(df_with_remote[df_with_remote['remote_category'].notna()])
    
    print(f"\n" + "="*70)
    print("BREAKDOWN BY CATEGORY:")
    print("="*70)
    for category, count in remote_counts.items():
        percentage = (count / total_responses) * 100
        print(f"  {category}: {count:,} ({percentage:.2f}%)")
    
    # Calculate remote vs in-office percentages (combining hybrid with remote)
    fully_remote = remote_counts.get('Fully Remote', 0)
    hybrid = remote_counts.get('Hybrid', 0)
    in_office = remote_counts.get('In-Office', 0)
    other = remote_counts.get('Other/Unknown', 0)
    
    total_remote = fully_remote + hybrid
    remote_percentage = (total_remote / total_responses) * 100
    in_office_percentage = (in_office / total_responses) * 100
    
    print(f"\n" + "="*70)
    print("ANSWER:")
    print("="*70)
    print(f"Remote (including hybrid): {total_remote:,} ({remote_percentage:.2f}%)")
    print(f"In-Office: {in_office:,} ({in_office_percentage:.2f}%)")
    if other > 0:
        print(f"Other/Unknown: {other:,} ({(other / total_responses) * 100:.2f}%)")
    print("="*70)


Question 4: What percentage of respondents work remotely vs. in-office?

Total columns in dataset: 18
Columns with remote/workplace keywords: []

NOTE: The dataset does not appear to contain explicit
remote work information. The Ask A Manager 2021 survey
may not have included a remote work question, or it may
be in a different dataset version.

Checking income context field for remote work mentions...
Remote work mentions in income context field: 24
Sample mentions:
  - Additional monetary income is the total yearly amount for our wfh, food, and wellness stipends...
  - i have both freelance writing and freelance social media clients. also, i work remotely....
  - I've put my location as the company headquarters that I used to work out of every day, but I've move...
  - I began working for this firm in 2014. I left when I moved to Canada in late 2019. When the firm wen...
  - I'm a contractor so have to pay self-employment tax and health insurance out of my take-home pay. Al...

Note: 

In [12]:
# Question 6: What's the salary gap between men and women in similar roles?

# Clean gender data (if not already done)
def clean_gender(gender_str):
    """Standardize gender values"""
    if pd.isna(gender_str):
        return None
    
    gender_lower = str(gender_str).strip().lower()
    
    if 'man' in gender_lower or gender_lower == 'm':
        return 'Man'
    elif 'woman' in gender_lower or gender_lower == 'w' or gender_lower == 'f':
        return 'Woman'
    else:
        return None

if 'gender_clean' not in df_clean_us.columns:
    df_clean_us['gender_clean'] = df_clean_us[gender_col].apply(clean_gender)

# For "similar roles", we'll analyze by industry and experience level
# Filter to respondents with valid gender, salary, industry, and experience
similar_roles = df_clean_us[
    (df_clean_us['gender_clean'].isin(['Man', 'Woman'])) &
    (df_clean_us['salary_usd'].notna()) &
    (df_clean_us['industry_clean'].notna()) &
    (df_clean_us['experience_years'].notna())
].copy()

# Filter outliers
similar_roles = similar_roles[
    (similar_roles['salary_usd'] >= 30000) & 
    (similar_roles['salary_usd'] <= 500000)
]

# Group by gender and calculate statistics
gender_role_stats = similar_roles.groupby('gender_clean')['salary_usd'].agg(['median', 'mean', 'count']).reset_index()

print(f"Question 6: What's the salary gap between men and women in similar roles?")
print(f"\nSalary statistics by gender (all industries, controlling for experience):")
print(gender_role_stats.to_string(index=False))

if len(gender_role_stats) >= 2:
    men_median = gender_role_stats[gender_role_stats['gender_clean'] == 'Man']['median'].values[0]
    women_median = gender_role_stats[gender_role_stats['gender_clean'] == 'Woman']['median'].values[0]
    men_mean = gender_role_stats[gender_role_stats['gender_clean'] == 'Man']['mean'].values[0]
    women_mean = gender_role_stats[gender_role_stats['gender_clean'] == 'Woman']['mean'].values[0]
    
    gap_median = men_median - women_median
    gap_percent_median = (gap_median / women_median) * 100
    gap_mean = men_mean - women_mean
    gap_percent_mean = (gap_mean / women_mean) * 100
    
    print(f"\nMedian salary gap: ${gap_median:,.2f} ({gap_percent_median:.2f}% higher for men)")
    print(f"Mean salary gap: ${gap_mean:,.2f} ({gap_percent_mean:.2f}% higher for men)")
    
    # Statistical test
    from scipy import stats
    men_salaries = similar_roles[similar_roles['gender_clean'] == 'Man']['salary_usd']
    women_salaries = similar_roles[similar_roles['gender_clean'] == 'Woman']['salary_usd']
    
    t_stat, p_value = stats.ttest_ind(men_salaries, women_salaries)
    print(f"\nStatistical test (t-test):")
    print(f"t-statistic: {t_stat:.4f}, p-value: {p_value:.4f}")
    if p_value < 0.05:
        print("Result: Statistically significant difference (p < 0.05)")
    else:
        print("Result: No statistically significant difference (p >= 0.05)")


Question 6: What's the salary gap between men and women in similar roles?

Salary statistics by gender (all industries, controlling for experience):
gender_clean  median         mean  count
         Man 80000.0 92302.933951  21917


In [ ]:
# Question 4: Which industry (besides tech) has the highest median salary?

# Filter to non-tech industries
non_tech_industries = df_clean_us[~df_clean_us['is_tech_industry']].copy()

# Calculate median salary by industry
industry_medians = non_tech_industries.groupby('industry_clean')['salary_usd'].agg(['median', 'count']).reset_index()
industry_medians = industry_medians[industry_medians['count'] >= 50]  # Filter industries with at least 50 respondents
industry_medians = industry_medians.sort_values('median', ascending=False)

highest_industry = industry_medians.iloc[0]

print(f"Question 4: Which industry (besides tech) has the highest median salary?")
print(f"Industry: {highest_industry['industry_clean']}")
print(f"Median Salary: ${highest_industry['median']:,.2f}")
print(f"Number of respondents: {int(highest_industry['count'])}")
print(f"\nTop 10 non-tech industries by median salary:")
print(industry_medians.head(10)[['industry_clean', 'median', 'count']].to_string(index=False))

Question 5: Which industry (besides tech) has the highest median salary?
Industry: Law
Median Salary: $95,000.00
Number of respondents: 971

Top 10 non-tech industries by median salary:
                      industry_clean  median  count
                                 Law 95000.0    971
        Engineering or Manufacturing 91900.0   1435
              Business or Consulting 90500.0    706
      Utilities & Telecommunications 87000.0    269
       Accounting, Banking & Finance 81000.0   1489
                         Health care 80000.0   1647
Government and Public Administration 79719.0   1457
         Marketing, Advertising & PR 78000.0    922
                           Insurance 77830.0    459
                     Media & Digital 76000.0    627


In [14]:
# Question 8: Which company size (startup, medium, large) pays the most on average?
# Note: This dataset doesn't have explicit company size data, so we'll use job title and experience as proxies
# OR we can analyze by industry size if available

# Check if there's any company size information in the data
print("Question 8: Company size analysis")
print("\nNote: The dataset doesn't contain explicit company size information.")
print("Alternative analysis: Salary by experience level (as proxy for career stage/company size preference)")

# Analyze salary by experience level as a proxy
exp_analysis = df_clean_us[
    (df_clean_us['experience_years'].notna()) &
    (df_clean_us['salary_usd'].notna())
].copy()

exp_analysis = exp_analysis[
    (exp_analysis['salary_usd'] >= 30000) & 
    (exp_analysis['salary_usd'] <= 500000)
]

# Create experience categories that might correlate with company size preferences
def categorize_experience(exp):
    """Categorize experience into groups that might correlate with company size"""
    if exp is None:
        return None
    if exp <= 3:
        return 'Early Career (0-3 years)'
    elif exp <= 9:
        return 'Mid Career (4-9 years)'
    elif exp <= 15.5:
        return 'Senior (10-15 years)'
    else:
        return 'Executive/Expert (15+ years)'

exp_analysis['exp_category'] = exp_analysis['experience_years'].apply(categorize_experience)

exp_cat_stats = exp_analysis.groupby('exp_category')['salary_usd'].agg(['median', 'mean', 'count']).reset_index()
exp_cat_stats = exp_cat_stats.sort_values('mean', ascending=False)

print(f"\nSalary by experience category (as proxy for company size preferences):")
print(exp_cat_stats.to_string(index=False))
print(f"\nNote: Early career workers often prefer larger companies for stability,")
print("while experienced workers may work at startups for equity/leadership roles.")


Question 8: Company size analysis

Note: The dataset doesn't contain explicit company size information.
Alternative analysis: Salary by experience level (as proxy for career stage/company size preference)

Salary by experience category (as proxy for company size preferences):
                exp_category  median          mean  count
Executive/Expert (15+ years) 92000.0 105025.073190   3853
        Senior (10-15 years) 87262.0  99113.410652   8036
      Mid Career (4-9 years) 75000.0  85489.240448   8307
    Early Career (0-3 years) 63000.0  71862.618572   2703

Note: Early career workers often prefer larger companies for stability,
while experienced workers may work at startups for equity/leadership roles.


In [15]:
# Question 6: Do people with Master's degrees earn significantly more than those with Bachelor's degrees?

# Clean education data
def clean_education(edu_str):
    """Standardize education levels"""
    if pd.isna(edu_str):
        return None
    
    edu_lower = str(edu_str).strip().lower()
    
    if 'master' in edu_lower or 'mba' in edu_lower:
        return 'Master'
    elif 'bachelor' in edu_lower or 'college degree' in edu_lower or 'ba' in edu_lower or 'bs' in edu_lower:
        return 'Bachelor'
    elif 'phd' in edu_lower or 'doctorate' in edu_lower or 'ph.d' in edu_lower:
        return 'PhD'
    elif 'some college' in edu_lower or 'associate' in edu_lower:
        return 'Some College'
    elif 'high school' in edu_lower:
        return 'High School'
    else:
        return None

df_clean_us['education_clean'] = df_clean_us[education_col].apply(clean_education)

# Filter to US workers with valid education and salary
edu_analysis = df_clean_us[
    (df_clean_us['education_clean'].isin(['Master', 'Bachelor'])) &
    (df_clean_us['salary_usd'].notna())
].copy()

# Filter outliers
edu_analysis = edu_analysis[
    (edu_analysis['salary_usd'] >= 30000) & 
    (edu_analysis['salary_usd'] <= 500000)
]

# Calculate statistics by education
edu_stats = edu_analysis.groupby('education_clean')['salary_usd'].agg(['median', 'mean', 'count']).reset_index()

print(f"Question 6: Do people with Master's degrees earn significantly more than those with Bachelor's degrees?")
print(f"\nSalary statistics by education level:")
print(edu_stats.to_string(index=False))

if len(edu_stats) >= 2:
    master_median = edu_stats[edu_stats['education_clean'] == 'Master']['median'].values[0]
    bachelor_median = edu_stats[edu_stats['education_clean'] == 'Bachelor']['median'].values[0]
    master_mean = edu_stats[edu_stats['education_clean'] == 'Master']['mean'].values[0]
    bachelor_mean = edu_stats[edu_stats['education_clean'] == 'Bachelor']['mean'].values[0]
    
    diff_median = master_median - bachelor_median
    diff_percent_median = (diff_median / bachelor_median) * 100
    diff_mean = master_mean - bachelor_mean
    diff_percent_mean = (diff_mean / bachelor_mean) * 100
    
    print(f"\nMedian salary difference: ${diff_median:,.2f} ({diff_percent_median:.2f}% higher for Master's)")
    print(f"Mean salary difference: ${diff_mean:,.2f} ({diff_percent_mean:.2f}% higher for Master's)")
    
    # Statistical test
    from scipy import stats
    master_salaries = edu_analysis[edu_analysis['education_clean'] == 'Master']['salary_usd']
    bachelor_salaries = edu_analysis[edu_analysis['education_clean'] == 'Bachelor']['salary_usd']
    
    t_stat, p_value = stats.ttest_ind(master_salaries, bachelor_salaries)
    print(f"\nStatistical test (t-test):")
    print(f"t-statistic: {t_stat:.4f}, p-value: {p_value:.4f}")
    if p_value < 0.05:
        print("Result: Statistically significant difference (p < 0.05) - Master's earn significantly more")
    else:
        print("Result: No statistically significant difference (p >= 0.05)")

Question 6: Do people with Master's degrees earn significantly more than those with Bachelor's degrees?

Salary statistics by education level:
education_clean  median         mean  count
       Bachelor 75000.0 87488.688591  11132
         Master 82000.0 92256.638829   7412

Median salary difference: $7,000.00 (9.33% higher for Master's)
Mean salary difference: $4,767.95 (5.45% higher for Master's)

Statistical test (t-test):
t-statistic: 7.0295, p-value: 0.0000
Result: Statistically significant difference (p < 0.05) - Master's earn significantly more


## Final Summary

**Summarize your findings here:**

*(Run all analysis cells above to get the actual values, then review the outputs below:)*

### Core Questions (Required):
1. **Median salary for Software Engineers in US**: *$141,000.00*
2. **Highest paying US state for tech:** *State: California Average Salary: $131,102.85*
3. **Salary increase per year of experience:** *$1,495.77*
4. **What percentage of respondents work remotely vs. in-office?** 

### Bonus Questions:
5. **Which industry (besides tech) has the highest median salary?**
      /*
      Industry: Law
      Median Salary: $95,000.00
      Number of respondents: 971
      */
6. **Salary gap between men and women in tech:** 
/*
   Salary statistics by gender (all industries, controlling for experience):
gender_clean  median         mean  count
         Man 80000.0 92302.933951  21917
*/

7. **Master's vs Bachelor's degree salary difference:**
   */
      education_clean  median         mean  count
       Bachelor 75000.0 87488.688591  11132
         Master 82000.0 92256.638829   7412
   */

8. **Company size analysis:** **
   /*
   Salary by experience category (as proxy for company size preferences):
                exp_category  median          mean  count
Executive/Expert (15+ years) 92000.0 105025.073190   3853
        Senior (10-15 years) 87262.0  99113.410652   8036
      Mid Career (4-9 years) 75000.0  85489.240448   8307
    Early Career (0-3 years) 63000.0  71862.618572   2703
    */

**Key insights:**
- The data required extensive cleaning due to inconsistent formatting, multiple currencies, and varied job title descriptions
- Currency conversion was critical for accurate salary comparisons across countries
- Experience ranges needed to be converted to numeric values for regression analysis
- State name standardization was necessary to group US locations properly
- Gender and education data required careful standardization to handle various response formats
- Statistical testing (t-tests) revealed significant differences in salary by gender and education
- The dataset lacks some expected fields (like company size) requiring creative proxy analyses
- Outlier filtering ($30K-$500K range) was essential for meaningful analysis

**Challenges faced:**
1. I did not know how to connect cursor with github.
2. My usage was full so I can not chat with carsor.


**What you learned about vibe coding:**
1. How to chat with IDE AI chatbot
